# Checking the communities with a second method (BERTopic)

The 13 communities in `project_Rakel.ipynb` were found with **one** method: TF-IDF vectors of the ingredient lists, cosine similarity, and Louvain on the full network. Modularity and the shuffle test show that the communities are *more grouped than chance*, but not that they are *right* (week 8: *"Modularity measures how grouped a partition is, not how right"*).

Here we group the same recipes with a **completely different method**, BERTopic (week 7), and compare the two groupings with NMI against shuffled labels (week 4 and 8).

| | Our communities | BERTopic |
|---|---|---|
| Recipe as | TF-IDF vector (exact ingredient names) | sentence embedding (meaning of the words) |
| Similarity | cosine, in a network | distance in the embedding space |
| Grouping | Louvain | UMAP + clustering |

**What this checks, and what it does not:** both methods use the same data (the clean ingredient lists), so this checks whether the communities depend on our *choice of method*, not whether our community names are right.

**Before running:** this notebook loads the files in `data/` made by `project_Rakel.ipynb`. The first run downloads the sentence-transformer model (about 90 MB).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

recipes = pd.read_parquet("data/recipes_clean.parquet")
communities = pd.read_parquet("data/communities.parquet")
recipes["ingr_clean"] = recipes["ingr_clean"].apply(list)   # parquet gives numpy arrays back
recipes = recipes.merge(communities, on="id")
print(f"Recipes: {len(recipes):,}, communities: {recipes['community'].nunique()}")

### Names of our communities

Same naming as in `reviews.ipynb`: the 4 ingredients that are most **over-represented** in each community (share of its recipes that use the ingredient, divided by the share of all recipes that use it).

In [ ]:
overall = recipes["ingr_clean"].explode().value_counts() / len(recipes)

community_name = {}
for c, group in recipes.groupby("community"):
    counts = group["ingr_clean"].explode().value_counts()
    counts = counts[counts >= min(20, max(2, len(group) // 2))]   # small communities: used by at least half
    lift = (counts / len(group)) / overall[counts.index]
    community_name[c] = ", ".join(lift.nlargest(4).index)

pd.set_option("display.max_colwidth", 80)
pd.Series(community_name, name="name").to_frame().assign(size=recipes["community"].value_counts())

## Step 1: each recipe as a text

BERTopic works on texts, so each recipe becomes its list of clean ingredients, separated by commas.

In [ ]:
docs = recipes["ingr_clean"].apply(", ".join).tolist()
docs[:3]

## Step 2: sentence embeddings

Each text becomes a vector of 384 numbers with the small sentence-transformer `all-MiniLM-L6-v2`. Texts with similar *meaning* get similar vectors, so unlike TF-IDF the model knows that *butter* and *margarine* are close.

Week 7 warns that long texts can be cut off without warning. The model reads at most 256 tokens, so we check that no recipe is longer.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
n_tokens = [len(model.tokenizer(d)["input_ids"]) for d in docs]
print(f"Longest recipe: {max(n_tokens)} tokens (limit {model.max_seq_length}), so nothing is cut off")

embeddings = model.encode(docs, show_progress_bar=False)
embeddings.shape

## Step 3: BERTopic

BERTopic makes the embeddings smaller with UMAP and then groups them. Two choices:

- **KMeans with 13 groups instead of the default HDBSCAN.** HDBSCAN puts many documents in an "outlier" group without a topic, and those recipes could not be compared. With 13 groups, the comparison with our 13 communities is direct.
- **A fixed `random_state`**, so the result can be repeated (we test other seeds in step 5).

**Topic names:** BERTopic's own topic words (class-based TF-IDF) remove punctuation and split the text into single words, so *baking soda* would become *baking* and *soda*. We therefore name the topics exactly like our communities: the 4 most over-represented ingredients. This also makes the names of the two groupings easy to compare.

In [ ]:
from bertopic import BERTopic
from umap import UMAP
from sklearn.cluster import KMeans
def run_bertopic(seed):
    topic_model = BERTopic(
        umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=seed),
        hdbscan_model=KMeans(n_clusters=13, n_init=10, random_state=seed))
    topics, _ = topic_model.fit_transform(docs, embeddings)
    return topic_model, np.array(topics)


topic_model, topics = run_bertopic(0)
recipes["topic"] = topics

def lift_names(group_column):
    # the 4 most over-represented ingredients in each group (as for the communities)
    names = {}
    for g, group in recipes.groupby(group_column):
        counts = group["ingr_clean"].explode().value_counts()
        counts = counts[counts >= min(20, max(2, len(group) // 2))]
        lift = (counts / len(group)) / overall[counts.index]
        names[g] = ", ".join(lift.nlargest(4).index)
    return names


topic_name = lift_names("topic")
pd.DataFrame({"name": topic_name, "size": recipes["topic"].value_counts()})

## Step 4: do the two methods find the same groups?

**NMI** (week 4) measures how much knowing a recipe's community tells us about its topic: 0 = nothing, 1 = the same grouping. On its own the number says little, so we compare it with **shuffled labels**: the communities shuffled between the recipes 1,000 times (same number and sizes of groups, but no link to the recipes).

In [ ]:
from sklearn.metrics import normalized_mutual_info_score as nmi

rng = np.random.default_rng(0)
real = nmi(recipes["community"], recipes["topic"])
null = np.array([nmi(rng.permutation(recipes["community"].to_numpy()), recipes["topic"]) for _ in range(1000)])
p = (1 + (null >= real).sum()) / (1 + len(null))

print(f"NMI between communities and BERTopic topics: {real:.3f}")
print(f"Shuffled communities: {null.mean():.4f} ± {null.std():.4f} (max {null.max():.4f}), p = {p:.3f}")

Which topic does each community end up in? Each row is one of our communities; the colour is the share of its recipes in each BERTopic topic. The topics are ordered so that each one comes after the community where most of its recipes are, so agreement shows up as a diagonal.

In [ ]:
share = pd.crosstab(recipes["community"], recipes["topic"], normalize="index")
sizes = recipes["community"].value_counts()
share = share.loc[sizes.index]                                               # biggest community first
best_row = pd.crosstab(recipes["community"], recipes["topic"]).loc[sizes.index].idxmax(axis=0)
order = sorted(share.columns, key=lambda t: list(sizes.index).index(best_row[t]))
share = share[order]

fig, ax = plt.subplots(figsize=(12, 7))
im = ax.imshow(share, cmap="Blues", vmin=0, vmax=1, aspect="auto")
ax.set_yticks(range(len(share)), [f"{community_name[c][:40]} ({sizes[c]})" for c in share.index])
ax.set_xticks(range(len(order)), [topic_name[t][:30] for t in order], rotation=60, ha="right")
ax.set_ylabel("Our community (Louvain on the TF-IDF network)")
ax.set_xlabel("BERTopic topic (sentence embeddings + UMAP + KMeans)")
fig.colorbar(im, label="share of the community's recipes")
ax.set_title(f"Two methods, the same recipes: NMI = {real:.2f} (shuffled: {null.mean():.3f})")
plt.tight_layout()
plt.show()

For each community: the topic where most of its recipes end up, and how big a share that is.

In [ ]:
best = pd.DataFrame({
    "community": [community_name[c] for c in share.index],
    "size": sizes[share.index].values,
    "best topic": [topic_name[t] for t in share.idxmax(axis=1)],
    "share in best topic": share.max(axis=1).round(2).values,
})
best

## Step 5: is BERTopic stable?

Louvain gave NMI 0.70 between runs with different seeds (`project_Rakel.ipynb`, step 9). We run BERTopic with 5 seeds and compare the runs with each other and with our communities.

In [ ]:
from itertools import combinations

runs = [topics] + [run_bertopic(seed)[1] for seed in [1, 2, 3, 4]]
between_runs = [nmi(a, b) for a, b in combinations(runs, 2)]
with_communities = [nmi(recipes["community"], r) for r in runs]
print(f"NMI between BERTopic runs:            mean {np.mean(between_runs):.2f} (min {min(between_runs):.2f}, max {max(between_runs):.2f})")
print(f"NMI between BERTopic and communities: mean {np.mean(with_communities):.2f} (min {min(with_communities):.2f}, max {max(with_communities):.2f})")

## What we found

- **The two methods agree far more than chance**: NMI = 0.38 between our communities and the BERTopic topics, against 0.007 ± 0.001 for shuffled communities (p = 0.001, 1,000 shuffles). This holds for all 5 BERTopic seeds (0.37–0.39).
- **The agreement is moderate, not complete.** For comparison, two BERTopic runs agree with NMI 0.88, and two Louvain runs with 0.70. So each method is fairly stable, and the two methods find partly different groupings.
- **The groups that our analysis builds on are found by both methods:**
  - *Yeast bread*: BERTopic has its own topic (*active yeast, bread flour, yeast, water*), and 41 % of our yeast-bread community ends up there.
  - *Asian* (70 % in one topic), *soups and stews* (46 %, topic *bay leaf, beef broth, celery, chicken stock*), *Mexican* (41 %, topic *taco seasoning, black beans, flour tortilla*) and *mac and cheese / casseroles* (43 %).
- **Where they disagree:**
  - BERTopic splits our big sweet-baking community (1,084 recipes) into several kinds of sweets: no-bake desserts (*cool whip, sweetened condensed milk*), quick breads (*baking soda, applesauce, banana*), cakes (*baking powder, almond extract*) and apple/oat desserts.
  - BERTopic finds a curry topic (*turmeric, chickpeas, curry powder*), which takes recipes from our Italian/Mediterranean and spice-rub communities.
  - This fits the difference between the methods: the embeddings know which ingredients *mean* similar things, while TF-IDF only matches exact names and gives rare ingredients much weight.

**For the report:** the communities are not just a product of TF-IDF + Louvain: a different method on the same recipes finds the same main dish types, in particular the yeast-bread, soup, Mexican and Asian groups that Question 1 is about. Limitation: both methods use the same ingredient lists, so this checks the method, not the names we give the communities.